# Exercise 8.3: Impact-parameter b-tagging

From *Programming in High Energy Particle Physics*, Chapter 8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch08/ex08_3_solution.ipynb)

Exercise 8.3 Impact-parameter b-tagging Build a simple b-tagging discriminator using track impact parameters. Generate simulated b-jets (with long-lived B hadron decays) and light-quark jets. Plot the signed impact parameter significance distributions and compute the ROC curve for discriminating b vs. light jets.

<details><summary>Hint</summary>

Give B hadrons an exponential decay length with \(c\tau \approx 0.45\) mm, boosted by \(\beta\gamma = p/m_B\) (\(m_B \approx 5.28\) GeV). The sign of \(d_0\) is positive if the point of closest approach lies in the jet direction. Use the second- or third-highest \(d_0/\sigma_{d_0}\) in the jet as a simple discriminant, and use the ROC tools of Chapter 9 .

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q numpy matplotlib scikit-learn
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)

### 1. Simulate prompt and displaced tracks

The transverse impact parameter is a simplified projection of B-hadron decay length. A small light-jet tail represents long-lived hadrons or tracking outliers; it is a toy model, not detector data.

In [ ]:
from sklearn.metrics import roc_curve,auc
mB=5.28 # GeV
ctau=0.45 # mm
sigma_d0=0.02 # mm
n_jets=4000
def make_jet(is_b,rng):
    n_tracks=int(rng.integers(10,17))
    sig=rng.normal(0,1,n_tracks)
    if is_b:
        pB=rng.uniform(30,100) # GeV
        Lxy=rng.exponential(ctau*pB/mB) # mm
        opening=rng.uniform(0.04,0.16,size=3)
        sig[:3]+=Lxy*np.sin(opening)/sigma_d0
    elif rng.random()<0.03:
        sig[0]+=rng.exponential(3)
    return sig
b_tracks=[make_jet(True,rng) for _ in range(n_jets)]
light_tracks=[make_jet(False,rng) for _ in range(n_jets)]
def tag(jets):return np.asarray([np.sort(t)[-2] for t in jets])
b_score,light_score=tag(b_tracks),tag(light_tracks)
y_true=np.r_[np.ones(n_jets),np.zeros(n_jets)]
score=np.r_[b_score,light_score]
fpr,tpr,thresholds=roc_curve(y_true,score)
roc_auc=auc(fpr,tpr)
print(f"AUC={roc_auc:.3f}")
assert roc_auc>0.5

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].hist(np.concatenate(b_tracks),bins=np.linspace(-5,30,80),density=True,histtype="step",label="b jets")
ax[0].hist(np.concatenate(light_tracks),bins=np.linspace(-5,30,80),density=True,histtype="step",label="light jets")
ax[0].set(xlabel=r"Signed $d_0/\sigma_{d_0}$",ylabel="Track density / bin");ax[0].legend()
ax[1].plot(fpr,tpr,label=f"AUC = {roc_auc:.3f}")
ax[1].plot([0,1],[0,1],"--",color="gray")
ax[1].set(xlabel="Light-jet mistag rate",ylabel="b-jet efficiency",xlim=(0,1),ylim=(0,1));ax[1].legend()
plt.tight_layout();plt.show()
idx=np.argmin(np.abs(fpr-0.01))
print(f"At light mistag {fpr[idx]:.3f}, b efficiency {tpr[idx]:.3f}")

The displaced tracks give a positive significance tail and separate b jets from prompt light jets. The ROC and AUC are properties of this simplified generator; realistic tagging needs track resolution, secondary vertices, charm and strange hadrons, pileup, and detector acceptance.